## 06. scratchpad

In [26]:
import pickle
import pandas as pd
from BPE_tokenizer import BPETokenizer, BPEEncoder
import Data
import re
from collections import Counter

In [2]:
with open("../data/dataframe_eng.pkl", 'rb') as f:
    data_eng = pickle.load(f)

with open("../data/dataframe_pol.pkl", 'rb') as f:
    data_pol = pickle.load(f)

In [5]:
with open("../data/bpe_tokenizer_eng.pkl", 'rb') as f:
    tokenizer_eng = pickle.load(f)
    
with open("../data/bpe_tokenizer_pol.pkl", 'rb') as f:
    tokenizer_pol = pickle.load(f)

In [56]:
data_pol['pol_ids'].str.len().nlargest(1000)

219180    46
6323      45
51920     45
3559      44
52078     44
          ..
5523      29
5567      29
5794      29
6629      29
6648      29
Name: pol_ids, Length: 1000, dtype: int64

In [33]:
df_final = pd.concat([data_eng, data_pol], axis=1)[['eng_text', 'eng_ids', 'pol_text', 'pol_ids']]

In [59]:
mask1 = df_final['eng_ids'].str.len() > 29
mask2 = df_final['pol_ids'].str.len() > 29
print(mask1.sum(), mask2.sum(), (mask1 | mask2).sum())
df_final = df_final[~(mask1 | mask2)].reset_index(drop=True)

1854 971 2338


In [65]:
data_test = df_final.sample(50000).reset_index(drop=True)
train_df, val_df = Data.shuffle_split(data_test, 0.85)

train_data = Data.EngPolDataset(train_df, 'eng_ids', 'pol_ids')
val_data = Data.EngPolDataset(val_df, 'eng_ids', 'pol_ids')

### New data

In [71]:
with open('../data/europarl/europarl-v7.pl-en.en', 'r', encoding='utf-8') as f:
    eng_lines = f.readlines()

with open('../data/europarl/europarl-v7.pl-en.pl', 'r', encoding='utf-8') as f:
    pol_lines = f.readlines()

pairs = [(e.strip(), p.strip()) for e, p in zip(eng_lines, pol_lines)
         if e.strip() and p.strip() 
         and not e.strip().startswith('<') 
         and not p.strip().startswith('<')]

df_europarl = pd.DataFrame(pairs, columns=['eng_text', 'pol_text'])
print(len(df_europarl))
df_europarl.head()

629380


,eng_text,pol_text
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół
2,Written statements (Rule 116): see Minutes,Oświadczenia pisemne (art. 116 Regulaminu): pa...
3,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...
4,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół


In [84]:
bool(re.search(r"[(]..[)]", "(IT) Mr President, the subject is an oral amen..."))

True

In [95]:
df_europarl[mask]['eng_text'].apply(lambda x: re.sub(r"^[(]..[)]", "", x))

60         It is proposed that the sum of 50 000 in para...
1656       Mr President, I would like to make an unequiv...
1713                              Mr. Chairman, colleagues,
1733       Mr President, after tens of years of politica...
1739       Mr President, a few months ago the European P...
                                ...                        
625096     Madam President, I was happy to back this rep...
625104     the better it will be for all the farmers of ...
625106     Madam President, the common agricultural poli...
625114     Madam President, ladies and gentlemen, I have...
628808     Mr President, ladies and gentlemen, the Europ...
Name: eng_text, Length: 18356, dtype: object

In [96]:
requir = lambda x: re.sub(r"^[(]..[)]", "", x)
df_europarl['eng_text'] = df_europarl['eng_text'].apply(requir)
df_europarl['pol_text'] = df_europarl['pol_text'].apply(requir)

In [108]:
bool(re.search(r".+[(].+[)].+", "Written statements (Rule 116): see."))

True

In [113]:
mask_eng = df_europarl['eng_text'].apply(lambda x: bool(re.search(r".+[(].+[)].+", x)))
mask_pol = df_europarl['pol_text'].apply(lambda x: bool(re.search(r".+[(].+[)].+", x)))

In [117]:
mask_eng.sum(), mask_pol.sum(), (mask_eng | mask_pol).sum()

(22461, 21196, 25231)

In [123]:
df_europarl = df_europarl[~(mask_eng | mask_pol)].reset_index(drop=True)
df_europarl

,eng_text,pol_text
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół
2,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...
3,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół
4,Membership of committees and delegations: see ...,Skład komisji i delegacji: patrz protokół
...,...,...
604144,Composition of committees and delegations : se...,Skład komisji i delegacji: Patrz protokól
604145,Agenda of the next sitting : see Minutes,Porządek obrad następnego posiedzenia: Patrz p...
604146,Closure of the sitting,Zamknięcie posiedzenia
604147,(The sitting closed at 22.25),(The sitting closed at 22.25)


In [125]:
uniq_eng = set("".join(df_europarl["eng_text"].astype(str)))
uniq_pol = set("".join(df_europarl["pol_text"].astype(str)))
print(f"Eng uniq: {len(uniq_eng)} | Pol uniq: {len(uniq_pol)}")

Eng uniq: 262 | Pol uniq: 267


In [163]:
tgt_eng = ['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '"', '!', ' ', ';']
tgt_pol = ['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '"', '!', ' ', ';']

In [164]:
eng_delchars = [x for x in sorted(list(uniq_eng), reverse=True) if x not in tgt_eng]
pol_delchars = [x for x in sorted(list(uniq_pol), reverse=True) if x not in tgt_pol]
regex_eng = "[" + "".join(map(re.escape, eng_delchars)) + "]"
regex_pol = "[" + "".join(map(re.escape, pol_delchars)) + "]"

In [168]:
eng_delchars

['$']

In [169]:
mask_eng = df_europarl["eng_text"].str.contains(regex_eng, na=False)
#mask_pol = df_europarl["pol_text"].str.contains(regex_pol, na=False)
mask_eng.sum()#, mask_pol.sum(), (mask_eng | mask_pol).sum()

3

In [172]:
#df_europarl = df_europarl[~(mask_eng | mask_pol)].reset_index(drop=True)
df_europarl = df_europarl[~mask_eng].reset_index(drop=True)

In [173]:
uniq_eng = set("".join(df_europarl["eng_text"].astype(str)))
uniq_pol = set("".join(df_europarl["pol_text"].astype(str)))
print(f"Eng uniq: {len(uniq_eng)} | Pol uniq: {len(uniq_pol)}")

Eng uniq: 78 | Pol uniq: 94


In [176]:
print(sorted(list(uniq_eng), reverse=True))

['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ';', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '"', '!', ' ']


In [181]:
df_europarl

,eng_text,pol_text
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół
2,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...
3,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół
4,Membership of committees and delegations: see ...,Skład komisji i delegacji: patrz protokół
...,...,...
579887,Composition of committees and delegations : se...,Skład komisji i delegacji: Patrz protokól
579888,Agenda of the next sitting : see Minutes,Porządek obrad następnego posiedzenia: Patrz p...
579889,Closure of the sitting,Zamknięcie posiedzenia
579890,(The sitting closed at 22.25),(The sitting closed at 22.25)


In [198]:
mask_1 = (df_europarl["eng_text"].str[-1] == '.') & (df_europarl["pol_text"].str[-1] != '.')
mask_2 = (df_europarl["eng_text"].str[-1] != '.') & (df_europarl["pol_text"].str[-1] == '.')

In [199]:
mask_1.sum(), mask_2.sum(), (mask_1 | mask_2).sum()

(1960, 1108, 3068)

In [201]:
(~(mask_1 | mask_2)).sum()

576824

In [202]:
df_europarl = df_europarl[~(mask_1 | mask_2)].reset_index(drop=True)

In [204]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [205]:
df_europarl['eng_split'] = df_europarl["eng_text"].apply(tokenize_eng)
df_europarl['pol_split'] = df_europarl["pol_text"].apply(tokenize_pol)

In [219]:
df_europarl['eng_split']

0         [action, taken, on, parliament, 's, resolution...
1                    [documents, received, :, see, minutes]
2         [texts, of, agreements, forwarded, by, the, co...
3             [membership, of, parliament, :, see, minutes]
4         [membership, of, committees, and, delegations,...
                                ...                        
576819    [composition, of, committees, and, delegations...
576820    [agenda, of, the, next, sitting, :, see, minutes]
576821                          [closure, of, the, sitting]
576822          [(, the, sitting, closed, at, 22, ., 25, )]
576823                                               [., -]
Name: eng_split, Length: 576824, dtype: object

In [230]:
uniq_freq_eng = Counter(df_europarl['eng_split'].explode())
uniq_freq_pol = Counter(df_europarl['pol_split'].explode())

In [266]:
char_factor = lambda word: [x for x in word] + ['_']
vocab_chars_eng = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
vocab_chars_pol = [[char_factor(k), v] for k, v in uniq_freq_pol.most_common() if v > 1]
print(len(vocab_chars_eng), vocab_chars_eng[:5], '\n')
print(len(vocab_chars_pol), vocab_chars_pol[:5]) 

50655 [[['t', 'h', 'e', '_'], 1019855], [[',', '_'], 712258], [['.', '_'], 562408], [['o', 'f', '_'], 488571], [['t', 'o', '_'], 458481]] 

100547 [[[',', '_'], 876176], [['.', '_'], 578767], [['w', '_'], 432544], [['i', '_'], 304762], [['n', 'a', '_'], 192303]]


In [267]:
pair_vocab_eng = [[list(zip(tab, tab[1:])), v] for tab, v in vocab_chars_eng]
pair_vocab_pol = [[list(zip(tab, tab[1:])), v] for tab, v in vocab_chars_pol]

In [254]:
import BPE_tokenizer
from tqdm.notebook import tqdm
import importlib
importlib.reload(BPE_tokenizer)

<module 'BPE_tokenizer' from 'C:\\Users\\Hubert\\Desktop\\eng-pol translator\\notebooks\\BPE_tokenizer.py'>

In [272]:
vocab_chars_eng, pair_vocab_eng

([[['t', 'h', 'e', '_'], 1019855],
  [[',', '_'], 712258],
  [['.', '_'], 562408],
  [['o', 'f', '_'], 488571],
  [['t', 'o', '_'], 458481],
  [['a', 'n', 'd', '_'], 409816],
  [['i', 'n', '_'], 328584],
  [['t', 'h', 'a', 't', '_'], 244130],
  [['i', 's', '_'], 232040],
  [['a', '_'], 230470],
  [['f', 'o', 'r', '_'], 165561],
  [['w', 'e', '_'], 162306],
  [['t', 'h', 'i', 's', '_'], 153705],
  [['i', '_'], 149372],
  [['o', 'n', '_'], 131780],
  [['i', 't', '_'], 125037],
  [['b', 'e', '_'], 115825],
  [['-', '_'], 113169],
  [['a', 'r', 'e', '_'], 106249],
  [['h', 'a', 'v', 'e', '_'], 96224],
  [['a', 's', '_'], 95598],
  [['e', 'u', 'r', 'o', 'p', 'e', 'a', 'n', '_'], 91656],
  [['n', 'o', 't', '_'], 90226],
  [['w', 'i', 't', 'h', '_'], 88256],
  [['w', 'i', 'l', 'l', '_'], 75079],
  [['w', 'h', 'i', 'c', 'h', '_'], 71357],
  [['b', 'y', '_'], 66418],
  [['h', 'a', 's', '_'], 58463],
  [['a', 'l', 's', 'o', '_'], 50414],
  [['a', 'n', '_'], 48353],
  [['a', 't', '_'], 48041],
  

In [269]:
tokenizer_eng = BPE_tokenizer.BPETokenizer(vocab_chars_eng, pair_vocab_eng, 12000, False)
tokenizer_pol = BPE_tokenizer.BPETokenizer(vocab_chars_pol, pair_vocab_pol, 24000, True)

In [273]:
tokenizer_eng.train_bpe()

  0%|          | 0/11945 [00:00<?, ?it/s]

In [274]:
tokenizer_pol.train_bpe()

  0%|          | 0/23935 [00:00<?, ?it/s]

In [45]:
encoder_eng = BPEEncoder(tokenizer_eng.vocab)
encoder_pol = BPEEncoder(tokenizer_pol.vocab, 65, True)

In [6]:
df_europarl.head()

,eng_text,pol_text,eng_split,pol_split,eng_ids
0,Action taken on Parliament's resolutions: see ...,Działania podjęte w wyniku rezolucji Parlament...,"[action, taken, on, parliament, 's, resolution...","[działania, podjęte, w, wyniku, rezolucji, par...","[771, 909, 73, 326, 258, 3994, 434, 790, 1791, 2]"
1,Documents received: see Minutes,Składanie dokumentów: patrz protokół,"[documents, received, :, see, minutes]","[składanie, dokumentów, :, patrz, protokół]","[2769, 2649, 434, 790, 1791, 2]"
2,Texts of agreements forwarded by the Council: ...,Teksty porozumień przekazane przez Radę: patrz...,"[texts, of, agreements, forwarded, by, the, co...","[teksty, porozumień, przekazane, przez, radę, ...","[4207, 78, 1458, 9696, 201, 61, 429, 434, 790,..."
3,Membership of Parliament: see Minutes,Skład Parlamentu: patrz protokół,"[membership, of, parliament, :, see, minutes]","[skład, parlamentu, :, patrz, protokół]","[3303, 78, 326, 434, 790, 1791, 2]"
4,Membership of committees and delegations: see ...,Skład komisji i delegacji: patrz protokół,"[membership, of, committees, and, delegations,...","[skład, komisji, i, delegacji, :, patrz, proto...","[3303, 78, 3853, 81, 5881, 434, 790, 1791, 2]"


In [42]:
from tqdm.auto import tqdm
tqdm.pandas()

In [47]:
df_2 = df_europarl[['pol_text', 'pol_split']].copy()
df_2['pol_ids'] = df_2['pol_text'].progress_apply(encoder_pol.encode_snt)

  0%|          | 0/576824 [00:00<?, ?it/s]

In [48]:
with open("../data/dataframe_pol_europarl.pkl", 'wb') as f:
    pickle.dump(df_2, f)

In [3]:
with open("../data/tokenizer_eng_2.pkl", 'rb') as f:
    tokenizer_eng = pickle.load(f)

with open("../data/tokenizer_pol_2.pkl", 'rb') as f:
    tokenizer_pol = pickle.load(f)

with open("../data/dataframe_europarl.pkl", 'rb') as f:
    df_europarl = pickle.load(f)

In [49]:
with open("../data/tokenizer_eng_2.pkl", 'wb') as f:
    pickle.dump(tokenizer_eng, f)

with open("../data/tokenizer_pol_2.pkl", 'wb') as f:
    pickle.dump(tokenizer_pol, f)

In [50]:
with open("../data/dataframe_europarl.pkl", 'wb') as f:
    pickle.dump(df_europarl, f)